# YARG Resurrected: convert a batch of songs

Turns a folder of songs in your Google Drive into YARG song folders, on Colab's GPU:
guitar and bass charts, and a vocals part with lyrics timed word by word against the voice.

**Lyrics:** looked up online (LRCLIB, NetEase, Lyrics.ovh) and timed from the singer's voice;
for songs with no lyrics anywhere, written from the voice by speech-to-text (Whisper large-v3,
about 3 words in 10 wrong on rap, fewer on singing).

**Before you start:** Runtime > Change runtime type > **GPU** (T4 or better). Put your songs in
`My Drive/YARG/input` (any of mp3, wav, ogg, flac, m4a; loose, in subfolders, or zipped). From a phone:
select the songs in your Files app, Share > Drive, and pick that folder.

**Optional, per song:** a `songs.csv` in that folder with columns
`file,mode,notes,title,artist,album,year,genre,start,end` (mode: split, rnb or normal; notes: more,
standard or fewer; start and end in seconds, to cut a music video's intro or outro off). Leave
columns empty to have them filled in automatically: title, artist, album, year and genre come from
the file's tags, its name ("Artist - Title") and MusicBrainz. For example:

```
file,mode,notes,title,artist,album,year,genre,start,end
Girls_Generation______Gee_MV.mp3,split,more,Gee,Girls' Generation,,,,19.9,220.5
Seduction.mp3,rnb,,,Usher,,,,,
```

Then **Runtime > Run all**. Results go to `My Drive/YARG/output`.

In [ ]:
#@title Settings
SONGS = "/content/drive/MyDrive/YARG/input"   #@param {type:"string"}
OUTPUT = "/content/drive/MyDrive/YARG/output" #@param {type:"string"}
MODE = "split"   #@param ["split", "rnb", "normal"]
NOTES = "more"   #@param ["more", "standard", "fewer"]
JOBS = "auto"    #@param ["auto", "1", "2", "3", "4"]
#@markdown JOBS: songs converted at once. "auto" picks from the runtime's memory and processors (an A100 or L4 runtime: usually 3 to 4; a T4: 1).
BRANCH = "ccr-33e397d3-vlla2w" #@param {type:"string"}

In [ ]:
#@title 1. Connect Google Drive
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
os.makedirs(SONGS, exist_ok=True); os.makedirs(OUTPUT, exist_ok=True)
AUDIO = (".mp3", ".wav", ".ogg", ".opus", ".flac", ".m4a", ".aac", ".webm")
# The songs are gathered on the Colab disk: loose files, files in subfolders, and the insides of zips
os.makedirs("/content/songs", exist_ok=True)
for f in os.listdir("/content/songs"): os.remove(os.path.join("/content/songs", f))
for root, _, files in os.walk(SONGS):
    for f in files:
        path = os.path.join(root, f)
        if f.lower().endswith(AUDIO):
            shutil.copy(path, "/content/songs/" + f)
        elif f.lower().endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                for name in z.namelist():
                    base = os.path.basename(name)
                    if base.lower().endswith(AUDIO) and not base.startswith("."):
                        with z.open(name) as src, open("/content/songs/" + base, "wb") as dst:
                            shutil.copyfileobj(src, dst)
        elif f == "songs.csv":
            shutil.copy(path, "/content/songs/songs.csv")
songs = sorted(f for f in os.listdir("/content/songs") if f.lower().endswith(AUDIO))
print(len(songs), "songs found in", SONGS, "(loose, in subfolders or in zips)")
for s in songs[:40]: print("  ", s)


**2. Set up** (about 3 minutes, once per session)

In [ ]:
%%bash -s "$BRANCH"
set -e
cd /content
apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless ffmpeg > /dev/null
pip -q install faster-whisper num2words
[ -d YARG_Resurrected ] || git clone -q --depth 1 -b "$1" https://github.com/ClockworkBlack/YARG_Resurrected.git
mkdir -p models lib
cd models
[ -s htdemucs_6s.onnx ] || curl -sSL --fail -o htdemucs_6s.onnx "https://huggingface.co/StemSplitio/htdemucs-6s-onnx/resolve/49df9b6989cf2150840ea65b0bef77a2e471b678/htdemucs_6s_fp16weights.onnx"
[ -s wav2vec2.onnx ] || curl -sSL --fail -o wav2vec2.onnx "https://huggingface.co/Xenova/wav2vec2-base-960h/resolve/a19f851b3d42865797e410752b4c570c871e4825/onnx/model.onnx"
echo "7ce55792e2231c93fbf92de95f5fd5b3a5e6c89f7db690dfd693e8f1dce56869  htdemucs_6s.onnx" | sha256sum -c
echo "e46614273f03ff4b87923a965e417fa72004825522cb007c9c25633b8475490d  wav2vec2.onnx" | sha256sum -c
cd ../lib
[ -s onnxruntime_gpu.jar ] || curl -sSL --fail -o onnxruntime_gpu.jar "https://repo1.maven.org/maven2/com/microsoft/onnxruntime/onnxruntime_gpu/1.31.1/onnxruntime_gpu-1.31.1.jar"
[ -s json.jar ] || curl -sSL --fail -o json.jar "https://repo1.maven.org/maven2/org/json/json/20240303/json-20240303.jar"
cd /content/YARG_Resurrected/converter
S=app/src/main/java/com/clockworkblack/yargconverter
rm -rf /content/build && javac -encoding UTF-8 -nowarn -cp /content/lib/onnxruntime_gpu.jar:/content/lib/json.jar -d /content/build \
  $(find $S -path '*core*' -name '*.java') $S/stems/*.java $S/LrcLib.java $S/NetEase.java $S/MusicBrainz.java \
  desktop/com/clockworkblack/yargconverter/Lookup.java core-test/MidiCheck.java desktop/ConvertSong.java 2>&1 | grep -v "^Note" || true
echo "Ready."
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: Runtime > Change runtime type > GPU"

In [ ]:
#@title 3. Convert
import os, subprocess
env = dict(os.environ,
           YARG_CP="/content/build:/content/lib/onnxruntime_gpu.jar:/content/lib/json.jar",
           YARG_STEMS="/content/models/htdemucs_6s.onnx", YARG_WORDS="/content/models/wav2vec2.onnx")
# Work on the Colab disk, then copy to Drive (much faster than writing to Drive song by song)
subprocess.run(["rm", "-rf", "/content/out"])
cmd = ["python", "/content/YARG_Resurrected/converter/desktop/batch_convert.py", "/content/songs", "/content/out",
       "--mode", MODE, "--notes", NOTES, "--gpu", "--jobs", JOBS]
with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for line in p.stdout:
        print(line, end="")

In [ ]:
#@title 4. Save to Drive (a zip to unzip into Internal storage/YARG/songs on the phone)
import shutil, time, os
stamp = time.strftime("%Y-%m-%d %H%M")
zip_path = shutil.make_archive(os.path.join(OUTPUT, "YARG songs " + stamp), "zip", "/content/out", "Converted")
shutil.copy("/content/out/report.csv", os.path.join(OUTPUT, "report " + stamp + ".csv"))
print("Saved", zip_path)
import pandas as pd
pd.read_csv("/content/out/report.csv")[["file", "folder", "lyrics", "vocalLines", "guitarExpert", "seconds", "error"]]

**On the phone:** unzip into `Internal storage/YARG/songs/`, then Settings > Songs > Scan Songs.

**report.csv** lists every song: where its lyrics came from (timed online, words online timed from
the voice, written by speech-to-text, or none), how many vocal lines, and anything that failed.
Lyrics written by speech-to-text are worth a look: they are in each song's `lyrics.lrc`.